## Load files

In [3]:
import json
import pandas as pd


## Annotating Paragraph

The annotation process made me realize that stance labeling is usually easier than relevance labeling, because many perspectives clearly support or oppose a claim, while relevance often sits on a spectrum and requires more judgment. I learned how important it is to separate agreement from topical fit. A perspective can be strongly worded and still be only somewhat relevant if it addresses a side issue rather than the core issue. It didn’t feel hard, just tedious. It was a little challenging when wording became vague and broad. A next iteration of the task would benefit from clearer annotation guidelines, especially with more edge-case examples showing the difference between “directly relevant” and “somewhat relevant”.  Overall, the exercise changed my perception of annotation by showing me that it is not just mechanical labeling; it is a careful interpretive process where good definitions and consistency matter as much as the labels themselves.

In [8]:
with open("perspectrum_with_answers_v1.0.json", "r", encoding="utf-8") as f:
    perspectrum_data = json.load(f)

with open("perspective_pool_v1.0.json", "r", encoding="utf-8") as f:
    perspective_pool_data = json.load(f)
    
annotated_df = pd.read_csv("annotated_sample_100.csv")
print(annotated_df.head())

   cid                                         claim_text    pid  \
0  942       Children should do part time and summer work  26954   
1  395  The United Nations has a responsibility to pro...   2940   
2  504         Encourage fewer people to go to university  24893   
3  564  The United States should continue Its use of d...   4191   
4  176                    Make all museums free of charge   1309   

                                    perspective_text stance_annotation  \
0  Children should work because a job provides sa...           support   
1  Blanket commitment creates a slippery slope of...            oppose   
2  Many people with college degrees find themselv...           support   
3  Drone strikes are subject to a strict review p...           support   
4  If state-funded, there is little incentive to ...            oppose   

  relevance_annotation  
0    directly relevant  
1    directly relevant  
2    directly relevant  
3    directly relevant  
4    directly relevan

## Setup
I set up a Python virtual environment on Great Lakes, installed the required packages, copied the dataset and annotation files to the cluster, and verified that PyTorch detected a GPU before training.

## Part 2


In [12]:
import json
import random
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

/Users/lamontenunn/Desktop/SI630/Homework3/.venv-1/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
SEED = 440
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

PERSPECTRUM_FILE = "perspectrum_with_answers_v1.0.json"
PERSPECTIVE_POOL_FILE = "perspective_pool_v1.0.json"
MODEL_NAME = "roberta-base"

## Load files

In [14]:
with open("perspectrum_with_answers_v1.0.json", "r", encoding="utf-8") as f:
    perspectrum_data = json.load(f)

with open("perspective_pool_v1.0.json", "r", encoding="utf-8") as f:
    perspective_pool_data = json.load(f)
    
    
print("Number of claims:", len(perspectrum_data))
print("Number of perspective pool items:", len(perspective_pool_data))

Number of claims: 907
Number of perspective pool items: 11112


In [15]:
pid_to_text = {}
for item in perspective_pool_data:
    pid_to_text[item["pId"]] = item["text"]

print("Lookup size:", len(pid_to_text))

Lookup size: 11112


In [16]:
def map_labels(stance_label_5):
    label = stance_label_5.strip().lower()

    if label == "support":
        return 1, 1
    elif label == "undermine":
        return 0, 1
    elif label == "mildly support":
        return 1, 0
    elif label == "mildly undermine":
        return 0, 0
    else:
        return None

In [ ]:
rows = []

for claim in perspectrum_data:
    cid = claim["cId"]
    claim_text = claim["text"]

    for perspective_group in claim["perspectives"]:
        pids = perspective_group["pids"]
        stance_label_5 = perspective_group["stance_label_5"]
        voter_counts = perspective_group["voter_counts"]

        # Drop not a valid perspective
        # identify these by nonzero last entry of voter_counts
        if len(voter_counts) > 4 and voter_counts[-1] > 0:
            continue

        # Drop no-majority cases
        if stance_label_5.strip().upper() == "NO MAJORITY LABEL":
            continue

        mapped = map_labels(stance_label_5)
        if mapped is None:
            continue

        stance_support, relevance_direct = mapped

        if not pids:
            continue

        # choose one representative pid from the group
        chosen_pid = random.choice(pids)
        perspective_text = pid_to_text.get(chosen_pid)

        if perspective_text is None:
            continue

        rows.append({
            "cid": cid,
            "claim_text": claim_text,
            "pid": chosen_pid,
            "perspective_text": perspective_text,
            "stance_support": stance_support,
            "relevance_direct": relevance_direct
        })

df = pd.DataFrame(rows).drop_duplicates()

print("Training rows after cleaning:", len(df))
print(df.head())

Training rows after cleaning: 4277
   cid                           claim_text    pid  \
0  499  Vaccination must be made compulsory   3695   
1  499  Vaccination must be made compulsory   3698   
2  499  Vaccination must be made compulsory  24080   
3  499  Vaccination must be made compulsory  24078   
4  499  Vaccination must be made compulsory   3699   

                                    perspective_text  stance_support  \
0   It is the state’s duty to protect its community                1   
1  Compulsory vaccination violates the individual...               0   
2  The impacts of unvaccinated populations stain ...               1   
3                        Children must be protected.               1   
4  It is a parental right to decide about vaccina...               0   

   relevance_direct  
0                 1  
1                 1  
2                 1  
3                 1  
4                 1  


In [18]:
df["labels"] = df.apply(
    lambda row: [float(row["stance_support"]), float(row["relevance_direct"])],
    axis=1
)

df[["claim_text", "perspective_text", "stance_support", "relevance_direct"]].head()

,claim_text,perspective_text,stance_support,relevance_direct
0,Vaccination must be made compulsory,It is the state’s duty to protect its community,1,1
1,Vaccination must be made compulsory,Compulsory vaccination violates the individual...,0,1
2,Vaccination must be made compulsory,The impacts of unvaccinated populations stain ...,1,1
3,Vaccination must be made compulsory,Children must be protected.,1,1
4,Vaccination must be made compulsory,It is a parental right to decide about vaccina...,0,1
